In [4]:
!hostname
!python -V
!pwd

77849d45efe1
Python 3.12.12
/content


In [5]:
!pip install docling


In [6]:
import torch
torch.cuda.is_available()

True

In [7]:
!pip install pathlib

In [8]:
from pathlib import Path
import logging
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import ThreadedPdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.threaded_standard_pdf_pipeline import ThreadedStandardPdfPipeline
import time

logger = logging.getLogger(__name__)

class MdProcessor:
    def __init__(self):
        # パイプライン設定（初期化時に1回だけ作る）
        pipeline_options = ThreadedPdfPipelineOptions(
            accelerator_options=AcceleratorOptions(
                device=AcceleratorDevice.CUDA,
            ),
            ocr_batch_size=4,
            layout_batch_size=64,
            table_batch_size=4,
            generate_picture_images=True,
            images_scale=2.0,
            do_ocr=True, # デジタルPDFならFalse推奨だが、今は検証用でTrue
        )

        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_cls=ThreadedStandardPdfPipeline,
                    pipeline_options=pipeline_options,
                )
            }
        )

    def convert(self, input_path: Path):
        """
        PDFを変換し、DoclingのDocumentオブジェクトを返す
        """
        logger.info(f"Converting PDF: {input_path}")
        try:
            result = self.converter.convert(input_path)
            if result.status != ConversionStatus.SUCCESS:
                raise Exception(f"Conversion failed with status: {result.status}")
            return result.document
        except Exception as e:
            logger.error(f"Docling conversion error: {e}")
            raise

In [9]:
!ls

docling_lines.md  docling.pdf  output  sample_data


In [10]:
!curl -o docling.pdf https://arxiv.org/pdf/2408.09869

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5436k  100 5436k    0     0  36.1M      0 --:--:-- --:--:-- --:--:-- 36.3M


In [11]:
from docling_core.types.doc import (
    PictureItem,
    TableItem,
    TextItem,
    SectionHeaderItem,
    ListItem,
    CodeItem,
    FormulaItem,
)

In [16]:
def main():
    # ---------------------------------------------------------
    # 1. 設定と準備
    # ---------------------------------------------------------
    base_dir = Path.cwd()
    # テスト用PDFのパス（実際運用時はここを引数などで変える）
    input_pdf =  base_dir / "docling.pdf"
    
    # 出力先の設定
    output_dir = base_dir / "output"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    images_dir = output_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    logger.info(f"Target PDF: {input_pdf}")

    # ---------------------------------------------------------
    # 2. インスタンス化（モデルロード）
    # ---------------------------------------------------------
    logger.info("Initializing modules...")
    
    # PDF解析エンジンの起動
    try:
        processor = MdProcessor()
    except Exception as e:
        logger.critical(f"Failed to initialize MdProcessor: {e}")
        return

    

    # ---------------------------------------------------------
    # 3. PDF解析の実行
    # ---------------------------------------------------------
    logger.info("Starting PDF conversion...")
    start_time = time.time()
    
    try:
        doc = processor.convert(input_pdf)
    except Exception as e:
        logger.error(f"PDF conversion failed: {e}")
        return

    logger.info(f"PDF parsed in {time.time() - start_time:.2f}s")

    # ---------------------------------------------------------
    # 4. 再構築ループ（画像保存 & 翻訳）
    # ---------------------------------------------------------
    logger.info("Starting translation and reconstruction loop...")

    md_lines = []
    image_counter = 0

    # ドキュメントを頭からお尻まで舐める
    for item, level in doc.iterate_items():

        # --- A. 画像 (保存してリンク) ---
        if isinstance(item, PictureItem):
            if item.image:
                image_counter += 1
                # ファイル名の生成
                filename = f"{input_pdf.stem}_img_{image_counter}.png"
                save_path = images_dir / filename

                # 実データの取得と保存
                pil_image = item.get_image(doc)
                if pil_image:
                    pil_image.save(save_path)
                    # Markdown用の相対パス
                    rel_path = f"./images/{filename}"
                    md_lines.append(f"\n![Image]({rel_path})\n")
                    logger.debug(f"Saved image: {filename}")
            else:
                # 画像枠はあるがデータがない場合
                md_lines.append("\n<!-- Empty Image Box -->\n")

        elif isinstance(item, CodeItem):
            md_lines.append(f"\n```{item.code_language}\n{item.text}\n```\n")

        elif isinstance(item, FormulaItem):
            md_lines.append(f"\n$$\n{item.text}\n$$\n")

        # --- B. 表 (Markdown化のみ) ---
        elif isinstance(item, TableItem):
            # 翻訳はリスクが高いので一旦そのまま
            md_lines.append(f"\n{item.export_to_markdown(doc=doc)}\n")


        # --- C. 見出し (翻訳) ---
        elif isinstance(item, SectionHeaderItem):
            prefix = "#" * (level + 1)
            # テキストを翻訳機に投げる
            md_lines.append(f"\n{prefix} {item.text}\n")

        # --- D. リスト (翻訳) ---
        elif isinstance(item, ListItem):
            md_lines.append(f"* {item.text}")

        # --- E. 本文 (翻訳) ---
        elif isinstance(item, TextItem):
            md_lines.append(f"{item.text}\n")
            # 進捗が見えるように少しログを出す
            if len(item.text) > 20:
                logger.info(f"Text translated ({len(item.text)} chars)")

    # ---------------------------------------------------------
    # 5. ファイル保存
    # ---------------------------------------------------------
    output_md = input_pdf.with_name(f'{input_pdf.stem}_translated.md')
    output_md_lines = input_pdf.with_name(f'{input_pdf.stem}_lines.md')
    try:
        
        with open(output_md_lines, "w", encoding="utf-8") as f:
            f.write("\n".join(md_lines))
        logger.info(f"SUCCESS! Markdown saved to: {output_md}")
    except IOError as e:
        logger.error(f"Failed to save file: {e}")

In [17]:
if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.StreamHandler()
        ]
    )
    main()

[INFO] 2025-11-20 07:53:24,114 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-20 07:53:24,155 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-11-20 07:53:24,156 [RapidOCR] torch.py:54: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-11-20 07:53:24,364 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-20 07:53:24,368 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2025-11-20 07:53:24,369 [RapidOCR] torch.py:54: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2025-11-20 07:53:24,450 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-11-20 07:53:24,524 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/di

In [15]:
item

NameError: name 'item' is not defined

In [ ]:
!ls

docling_lines.md  docling.pdf  output  sample_data


In [ ]:
!cd output && ls
!cd images && ls

images
/bin/bash: line 1: cd: images: No such file or directory


In [ ]:
!cat notes docling_lines.md

cat: notes: No such file or directory

![Image](./images/docling_img_1.png)


## Docling Technical Report


## Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research R¨ uschlikon, Switzerland


## Abstract

This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.


## 1 Introduction

Converting PDF documents back into a machine-processable for